# 11 — Real Toronto sensory-dimension data: first exploration

First real step toward `sensory_weight()`: pulling real City of Toronto open data (not synthetic, not OSM tags) for the crowding dimension, starting with the **Traffic and Pedestrian Volumes** dataset. Construction data (Road Reconstruction, Road Resurfacing, Sidewalk Construction, Utility Cut Permits) follows the same two-step CKAN pattern once this one works.

See `research/sensory-accessibility-ontario.md` for the full research and the plan behind this module.

### Step 1 — get the dataset's package info

CKAN's `package_show` returns metadata and a list of `resources` — not the actual rows yet. Find the real dataset name (the `id` param) for pedestrian/traffic volumes on the Toronto Open Data catalogue, call `package_show`, and print each resource's `id`, `name`, and `format` so we know which resource to query next.

In [ ]:
import requests
import sys
sys.path.append("..")

from src.graph.snapping import nearest_node_id

BASE_URL = "https://ckan0.cf.opendata.inter.prod-toronto.ca/api/3/action"

respound = requests.get(f"{BASE_URL}/package_show", params={"id": "traffic-volumes-at-intersections-for-all-modes"})
package = respound.json()["result"]

for resource in package["resources"]:
    print(resource["id"], "|", resource["name"], "|", resource["format"])


### Step 2 — pull real rows with `datastore_search`

Pick the resource id that actually holds tabular data (not a PDF/ZIP), and call `datastore_search` with a small `limit` first — we just want to see the real column names and a few sample rows before pulling everything.

In [2]:
resource_id = "6afa3b1f-f6a5-4235-8bd6-7568411c19f4"
resp = requests.get(f"{BASE_URL}/datastore_search", params={"id": resource_id, "limit": 5})
rows = resp.json()["result"]["records"]
for row in rows:
    print(row)

{'_id': 1, 'latest_count_id': 117984, 'latest_count_date': '2026-10-27', 'location_name': 'Scarborough Golf Club Rd / Bankwell Ave / Slan Ave', 'longitude': -79.2200726425321, 'latitude': 43.7732636699038, 'centreline_type': '2', 'centreline_id': 13446602, 'px': 2174, 'count_duration': '14', 'total_vehicle': 7257, 'total_bike': 51, 'total_heavy_pct': 0.022, 'total_pedestrian': 226, 'am_peak_start': '2026-10-27T08:15:00', 'am_peak_vehicle': 639, 'am_peak_bike': 4, 'am_peak_heavy_pct': 0.033, 'pm_peak_start': '2026-10-27T15:00:00', 'pm_peak_vehicle': 688, 'pm_peak_bike': 8, 'pm_peak_heavy_pct': 0.029, 'n_appr_vehicle': 2943, 'n_appr_bike': 16, 'n_appr_heavy_pct': 0.022, 'e_appr_vehicle': 540, 'e_appr_bike': 13, 'e_appr_heavy_pct': 0.026, 's_appr_vehicle': 3559, 's_appr_bike': 15, 's_appr_heavy_pct': 0.02, 'w_appr_vehicle': 215, 'w_appr_bike': 7, 'w_appr_heavy_pct': 0.042}
{'_id': 2, 'latest_count_id': 117986, 'latest_count_date': '2026-10-27', 'location_name': 'Lawrence Ave E / Scarborou

### Step 3 — what do we actually have?

Look at the real fields: is there a geographic location (lat/lon or an intersection name) we can match to OSM nodes/edges? Is there an actual volume/count number, and for what time period? Print the full field list and a few full rows, not just a glance — this decides whether the dataset is usable as-is or needs a different resource.

In [3]:
resp = requests.get(f"{BASE_URL}/datastore_search", params={"id": resource_id, "limit": 7000})
result = resp.json()["result"]
rows = result["records"]
print("rows fetched:", len(rows), "of", result["total"])

pedestrian_counts = [row["total_pedestrian"] for row in rows if row["total_pedestrian"] is not None]

import statistics
print("min:", min(pedestrian_counts))
print("max:", max(pedestrian_counts))
print("median:", statistics.median(pedestrian_counts))
quantiles = statistics.quantiles(pedestrian_counts, n=4)
print("quartiles (25/50/75):", quantiles)


rows fetched: 6421 of 6421
min: 0
max: 91808
median: 425
quartiles (25/50/75): [164.0, 425.0, 1197.5]


### Step 4 — bucket the real pedestrian counts

Use `pandas.cut()` with the real quartile boundaries (164, 1197.5) from the full 6,421-row dataset, instead of a hand-written if/elif chain.

In [4]:
import pandas as pd

crowding_bucket = pd.cut(
    pedestrian_counts,
    bins=[-1, 164, 1197.5, float("inf")],
    labels=["low", "medium", "high"]
)

print(pd.Series(crowding_bucket).value_counts())


medium    3200
low       1616
high      1605
Name: count, dtype: int64


### Step 5 — a reusable crowding-level function for an edge

Averages the raw pedestrian counts of an edge's two endpoint nodes first (reduces per-point noise), then buckets the average with the same real quartile thresholds.

In [5]:
from src.graph.sensory_weighting import crowding_level

# quick check with two real counts from the sample rows earlier
print(crowding_level(226, 3330))   # low + high endpoints -> averaged, then bucketed
print(crowding_level(50, 80))      # two quiet endpoints
print(crowding_level(2000, 5000))  # two busy endpoints


high
low
high


### Step 6 — snap a real Toronto intersection to the nearest OSM node

Generalized version of the toy example: works on any `osmnx`-style graph `G`, where every node has `"y"` (latitude) and `"x"` (longitude) attributes -- the same convention `accessibility_weight()` already relies on.

In [ ]:
from src.graph.snapping import nearest_node_id

# Downtown Toronto's rough bounding box -- find a real intersection
# actually inside the graph's coverage area, not a random row.
downtown_rows = [
    row for row in rows
    if 43.63 <= row["latitude"] <= 43.68 and -79.41 <= row["longitude"] <= -79.36
]
print(len(downtown_rows), "rows inside the Downtown Toronto bounding box")

sample_row = downtown_rows[0]
target = (sample_row["latitude"], sample_row["longitude"])
print("target intersection:", sample_row["location_name"], target)


### Step 7 — load a real Toronto graph to snap onto

Same `osmnx` settings as Module 6 (Downtown Toronto, cached) -- reusing the already-downloaded area so this doesn't trigger a fresh, slow Overpass download.

In [7]:
import osmnx as ox

ox.settings.cache_folder = "cache"
ox.settings.useful_tags_way += ["wheelchair", "surface"]
ox.settings.useful_tags_node += ["kerb", "crossing", "tactile_paving"]
ox.settings.overpass_url = "https://overpass.kumi.systems/api/interpreter"

G = ox.graph_from_place("Downtown Toronto, Toronto, Ontario, Canada", network_type="walk")
print(G.number_of_nodes(), "nodes")

nearest_id = nearest_node_id(G, target, max_distance_m=200)
if nearest_id is None:
    print("no node within 200m -- this point is outside the graph's coverage")
else:
    print("nearest OSM node to", sample_row["location_name"], ":", nearest_id)
    print("node coords:", G.nodes[nearest_id]["y"], G.nodes[nearest_id]["x"])


14123 nodes
nearest OSM node to Brunswick Ave / Barton Ave : 58365732
node coords: 43.6689108 -79.4086507


### Step 8 — crowding level for a real graph edge

Pick a real edge from `G`, snap each endpoint (`u`, `v`) to the nearest Toronto pedestrian-volume record with `nearest_row()`, then run both counts through `crowding_level()`. Requires both endpoints to have a real match within `max_distance_m` -- if either doesn't, we have no real crowding data for this edge and should say so, not guess.

In [ ]:
from src.graph.sensory_weighting import edge_crowding

# try it on a real edge near our matched intersection
u, v, k = list(G.edges(keys=True))[0]
print("edge:", u, "->", v)
print("crowding level:", edge_crowding(G, u, v, downtown_rows))
